# Project 2: The Relationship Between Bachelor's Degree Production and College Graduate Underemployment Rates in the U.S.
### Welcome to my second project!
In this notebook, I explore whether two different datasets related to U.S. education might have something interesting to say when paired together.

### Research Question (Initial Idea)
I started with a simple question:
As more students graduate with bachelor's degrees each year, are they finding it harder to get jobs that actually require a college degree?
This is a question that hits close to home for many of us. We hear all the time about how "everyone needs a college degree," but we also hear stories about graduates working jobs that don't really need that expensive piece of paper. So I wanted to dig into the data and see if there's actually a relationship between how many bachelor's degrees are awarded and the underemployment rate.

**Hypothesis:** I suspect that as more people get bachelor's degrees, the underemployment rate might go up because the job market gets more saturated. But let's see what the data actually says!


### Data Sources
I'm using two datasets for this analysis:

1. Federal Underemployment Rates Dataset("College-labor-data.csv"): This comes from the Federal Reserve Bank of New York and tracks underemployment rates for college graduates from 1990-2025. https://www.newyorkfed.org/research/college-labor-market#--:explore:underemployment 

2. Graduates Census Dataset ("tabn318.10.csv"): This is from the National Center for Education Statistics and shows the number of degrees conferred by U.S. institutions from 1870-2032 (projections included). https://nces.ed.gov/programs/digest/d23/tables/dt23_318.10.asp

Let's dive in!

### Part 1: Cleaning the federal unemployment rates dataset

The underemployment rate captures the proportion of college-educated workers who end up in positions that typically do not require a four-year degree. A job counts as “college-level” only when at least half of the people in that occupation report that a bachelor’s degree is needed; otherwise, it falls into the non-college category.

All rates are seasonally adjusted and then smoothed using a three-month moving average. “College graduates” refers to adults aged 22–65 who hold at least a bachelor’s degree, while “recent graduates” are defined as those aged 22–27 with the same educational level. Individuals still enrolled in school are not included in the data. 

To get started, I need to clean up the underemployment data. The CSV has monthly data, but I want to look at yearly trends to match with the degree production data.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
import pandas as pd
import plotly.express as px
# ensure the visualizations render properly across Vscode, Jupyter Book, etc.
# https://plotly.com/python/renderers/

In [2]:
# Load the CSV file
# Tried to display the first few rows to see what the data looks like
# Found that the first 9 rows were title, so skipped them
df = pd.read_csv('College-labor-data.csv', skiprows=9)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               426 non-null    object 
 1   Recent graduates   426 non-null    float64
 2   College graduates  426 non-null    float64
dtypes: float64(2), object(1)
memory usage: 10.1+ KB


,Date,Recent graduates,College graduates
0,1990-01-01,42.9,34.1
1,1990-02-01,43.2,34.1
2,1990-03-01,43.3,34.1
3,1990-04-01,44.5,34.2
4,1990-05-01,44.8,34.3


In [3]:
df['Date'] = pd.to_datetime(df['Date'])

# Extract year from the 'Date' column
df['Year'] = df['Date'].dt.year

# Group by year and calculate the average of 'Recent graduates' and 'College graduates'
yearly_avg = df.groupby('Year')[['Recent graduates', 'College graduates']].mean().reset_index()

# Round to one decimal place
yearly_avg = yearly_avg.round(1)
print(yearly_avg)

# Save the results to a new CSV file
yearly_avg.to_csv('yearly_average.csv', index=False)

    Year  Recent graduates  College graduates
0   1990              44.1               34.0
1   1991              44.4               33.7
2   1992              47.5               34.5
3   1993              46.5               34.3
4   1994              46.5               34.0
5   1995              44.4               32.7
6   1996              43.2               32.9
7   1997              41.6               32.5
8   1998              40.6               32.5
9   1999              39.5               31.9
10  2000              38.3               31.7
11  2001              37.8               31.8
12  2002              38.5               32.4
13  2003              41.4               33.5
14  2004              44.4               34.1
15  2005              43.1               34.3
16  2006              43.2               34.4
17  2007              42.0               34.0
18  2008              42.6               33.8
19  2009              44.2               34.0
20  2010              45.4        

A few takeaways:
- I now have clean yearly underemployment rates from 1990-2025, group by the year. 
- Recent graduates consistently have higher underemployment rates than all college graduates (makes sense - it's harder when you're just starting out)
- The rates fluctuate but seem to hover in the 30-47% range. That's pretty high!

### Part 2: Cleaning the graduates census dataset

Now for the degree production data. This dataset is messier because it covers 150+ years and has multiple degree types. I only care about bachelor's degrees.

In [4]:
# Load the CSV file
# Tried to display the first few rows to see what the data looks like
# Found that the first 1 row was title, so skipped them
df2 = pd.read_csv('tabn318.10.csv', skiprows=1)

df2.info()
df2.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Year                   71 non-null     object
 1   Associate's degrees    65 non-null     object
 2   Associate's degrees.1  65 non-null     object
 3   Associate's degrees.2  65 non-null     object
 4   Associate's degrees.3  65 non-null     object
 5   Bachelor's degrees     65 non-null     object
 6   Bachelor's degrees.1   65 non-null     object
 7   Bachelor's degrees.2   65 non-null     object
 8   Bachelor's degrees.3   65 non-null     object
 9   Master's degrees       65 non-null     object
 10  Master's degrees.1     65 non-null     object
 11  Master's degrees.2     65 non-null     object
 12  Master's degrees.3     65 non-null     object
 13  Doctor's degrees\1\    65 non-null     object
 14  Doctor's degrees\1\.1  65 non-null     object
 15  Doctor's degrees\1\.2  65

,Year,Associate's degrees,Associate's degrees.1,Associate's degrees.2,Associate's degrees.3,Bachelor's degrees,Bachelor's degrees.1,Bachelor's degrees.2,Bachelor's degrees.3,Master's degrees,Master's degrees.1,Master's degrees.2,Master's degrees.3,Doctor's degrees\1\,Doctor's degrees\1\.1,Doctor's degrees\1\.2,Doctor's degrees\1\.3
0,Year,Total,Male,Female,Percent female,Total,Male,Female,Percent female,Total,Male,Female,Percent female,Total,Male,Female,Percent female
1,1869-70,---,---,---,---,"9,371","7,993","1,378",14.7,0,0,0,---,1,1,0,0
2,1879-80,---,---,---,---,"12,896","10,411","2,485",19.3,879,868,11,1.3,54,51,3,5.6
3,1889-90,---,---,---,---,"15,539","12,857","2,682",17.3,"1,015",821,194,19.1,149,147,2,1.3
4,1899-1900,---,---,---,---,"27,410","22,173","5,237",19.1,"1,583","1,280",303,19.1,382,359,23,6


**Challenge spotted:** The year column has ranges like "1869-70" and "1899-1900". I need to extract just the ending year to have a single year value. Also, there are multiple columns for each degree type (total, male, female, percent female) - I only need the total bachelor's degrees.

In [5]:
def get_end_year_safe(year_range):
    year_range = str(year_range).strip()
    try:
        year_range = str(year_range).strip()
        
        # Exclude invalid entries
        if not any(c.isdigit() for c in year_range):
            return None
            
        if '-' not in year_range:
            return int(year_range)
            
        parts = year_range.split('-')
        start_year = parts[0]
        end_year = parts[1]
            
        if len(end_year) == 2:
            end_year = start_year[:2] + end_year
            
        return int(end_year)
    
    except (ValueError, IndexError, AttributeError):
        # Handle cases where conversion fails or format is unexpected
        return None

# Apply the function to the 'Year' column
df2['Year'] = df2['Year'].apply(get_end_year_safe)
df2 = df2.dropna(subset=['Year'])
df2['Year'] = df2['Year'].astype(int)

df2.head()

,Year,Associate's degrees,Associate's degrees.1,Associate's degrees.2,Associate's degrees.3,Bachelor's degrees,Bachelor's degrees.1,Bachelor's degrees.2,Bachelor's degrees.3,Master's degrees,Master's degrees.1,Master's degrees.2,Master's degrees.3,Doctor's degrees\1\,Doctor's degrees\1\.1,Doctor's degrees\1\.2,Doctor's degrees\1\.3
1,1870,---,---,---,---,"9,371","7,993","1,378",14.7,0,0,0,---,1,1,0,0
2,1880,---,---,---,---,"12,896","10,411","2,485",19.3,879,868,11,1.3,54,51,3,5.6
3,1890,---,---,---,---,"15,539","12,857","2,682",17.3,"1,015",821,194,19.1,149,147,2,1.3
4,1900,---,---,---,---,"27,410","22,173","5,237",19.1,"1,583","1,280",303,19.1,382,359,23,6
5,1910,---,---,---,---,"37,199","28,762","8,437",22.7,"2,113","1,555",558,26.4,443,399,44,9.9


In [6]:
# Extract relevant columns
bachelor_data = df2.iloc[:, [0, 5]]  # Extracting 'Year' and 'Bachelor's total' columns
bachelor_data.columns = ['Year', 'Bachelor_Total']  # Renaming columns

# Drop rows with NaN values in 'Bachelor_Total'
bachelor_data = bachelor_data[bachelor_data['Bachelor_Total'].notna()]

print(bachelor_data)

# Save the results to a new CSV file
bachelor_data.to_csv('bachelor_total_by_year.csv', index=False)

    Year Bachelor_Total
1   1870          9,371
2   1880         12,896
3   1890         15,539
4   1900         27,410
5   1910         37,199
..   ...            ...
60  2028      2,319,984
61  2029      2,363,718
62  2030      2,402,215
63  2031      2,434,333
64  2032      2,464,382

[64 rows x 2 columns]


A few takeaways:

- Successfully extracted bachelor's degree totals from 1870-2032.
- The growth is dramatic! From 9,371 degrees in 1870 to over 2 million projected in 2025.
- The commas in the numbers will need to be handled when I merge the datasets.

### Part 3: Merging the two datasets

Now comes the crucial step - combining these datasets. 
I can only analyze years where I have both underemployment data AND degree production data. Looking at my ranges:
- Underemployment data: 1990-2025
- Degree data: 1870-2032

The overlap is 1990-2025, but I'm going to focus on 2016-2025 to look at the most recent trends.

In [7]:
# Merge the two datasets on the 'Year' column
merged_df = pd.merge(bachelor_data, yearly_avg, on='Year', how='inner')
# Filter for years greater than 2015
merged_df = merged_df[merged_df['Year'] > 2015]

print('Merged Data for Plotting (Year > 2015)')
merged_df

Merged Data for Plotting (Year > 2015)


,Year,Bachelor_Total,Recent graduates,College graduates
26,2016,"1,920,750",44.0,34.4
27,2017,"1,956,114",43.6,34.5
28,2018,"1,980,665",41.6,34.2
29,2019,"2,013,086",41.5,34.1
30,2020,"2,038,682",41.0,33.1
31,2021,"2,066,463",40.9,33.5
32,2022,"2,015,035",40.3,33.5
33,2023,"2,114,168",39.7,33.1
34,2024,"2,136,037",40.3,32.9
35,2025,"2,167,569",40.8,33.4


**What I'm looking at now:**
- 10 years of data (2016-2025) which are more recent
- Bachelor's degrees awarded each year
- Underemployment rates for both recent grads and all college grads

Initial observation: Bachelor's degree production went from 1.9 million in 2016 to 2.2 million in 2025. That's about a 12% increase. Did underemployment rates increase too? Let's visualize!

### Part 4: Data visualization

I'll use a bar chart for degree production (since it's a count) and line plots for underemployment rates (since they're percentages).

In [8]:
# Convert Bachelor_Total from string to number and scale to thousands for readability
merged_df["Bachelor_Total_scaled"] = merged_df["Bachelor_Total"].str.replace(",", "").astype(int) / 1000
merged_df

,Year,Bachelor_Total,Recent graduates,College graduates,Bachelor_Total_scaled
26,2016,"1,920,750",44.0,34.4,1920.750
27,2017,"1,956,114",43.6,34.5,1956.114
28,2018,"1,980,665",41.6,34.2,1980.665
29,2019,"2,013,086",41.5,34.1,2013.086
30,2020,"2,038,682",41.0,33.1,2038.682
31,2021,"2,066,463",40.9,33.5,2066.463
32,2022,"2,015,035",40.3,33.5,2015.035
33,2023,"2,114,168",39.7,33.1,2114.168
34,2024,"2,136,037",40.3,32.9,2136.037
35,2025,"2,167,569",40.8,33.4,2167.569


In [ ]:
# Create the combined visualization
fig = px.bar(
    merged_df,
    x="Year",
    y="Bachelor_Total_scaled",
    title="Total Bachelor's degrees conferred in the U.S. in thousands (Bar) + Underemployment rate in the U.S. (Line)"
)

# Add line for recent graduates' underemployment rate
fig.add_scatter(
    x=merged_df["Year"],
    y=merged_df["Recent graduates"],
    mode="lines+markers",
    name="Recent college graduates' underemployment rate (%)",
    yaxis="y2"
)

# Add line for all college graduates' underemployment rate
fig.add_scatter(
    x=merged_df["Year"],
    y=merged_df["College graduates"],
    mode="lines+markers",
    name="College graduates' underemployment rate (%)",
    yaxis="y2"
)

# Configure dual y-axes
fig.update_layout(
    yaxis2=dict(overlaying='y', side='right')
)

fig.show()

# Save the figure as an HTML file
fig.write_html("underemployment_chart.html", include_plotlyjs='cdn')


## Final Analysis and Takeaways

**What the visualization shows:**

1. **Bachelor's degree production is growing** (blue bars): From about 1.9 million in 2016 to 2.2 million in 2025 - that's roughly 250,000 more graduates per year!

2. **Underemployment rates are actually declining** (orange and green lines): 
   - Recent graduates: dropped from 44% in 2016 to about 40% in 2025
   - All college graduates: dropped from 34.4% to 33.4%

3. **The surprising finding:** Despite MORE people getting bachelor's degrees, the underemployment rate is going DOWN, not up!

**Why might this be happening?**
- The economy might be creating college-level jobs faster than we're producing graduates
- The COVID-19 pandemic (2020) created a temporary spike in underemployment, but things recovered
- The job market might actually need even more college graduates than we're producing

**My initial hypothesis was wrong!** I expected to see underemployment increase with degree production, but the data shows the opposite. This suggests that getting a bachelor's degree is still a good investment - the market can absorb more graduates.

**Limitations to consider:**
- This only covers 2016-2025 (recent years)
- Underemployment doesn't mean unemployment - these grads have jobs, just not ones requiring their degree
- We can't see field-specific trends (STEM vs. humanities, for example)
- Economic conditions vary significantly across this period

**Bottom line:** If you're worried that "everyone has a degree now" and that will hurt your job prospects, this data suggests that concern might be overblown. The job market seems to be keeping pace with the growing number of graduates.

## Potential Future Analysis

Questions I'd love to explore with more data:
- How do underemployment rates vary by major/field of study?
- What types of jobs are college graduates taking when they're "underemployed"?
- How do these trends compare internationally?
- What's the relationship with student debt levels?

Thanks for reading!